# Unidad 7 · Modelo lineal y ANOVA de un factor

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (reparto en
bicicleta) y la pieza del caso Estampa Sur (secado según tinta). Los datos están en `../datos/`: el ejemplo
en `u07_ejemplo.csv` (lo genera `verificacion/u07_anova_un_factor.py`) y el caso en `secado.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

DATOS = Path("..") / "datos"

## 1. Ejemplo · la tabla ANOVA a mano y con statsmodels

In [ ]:
df = pd.read_csv(DATOS / "u07_ejemplo.csv")
grupos = {g: v.minutos.to_numpy(float) for g, v in df.groupby("bicicleta", sort=False)}

# a mano
media_global = df.minutos.mean()
sc_trat = sum(len(v) * (v.mean() - media_global) ** 2 for v in grupos.values())
sc_error = sum(((v - v.mean()) ** 2).sum() for v in grupos.values())
k, N = len(grupos), len(df)
F = (sc_trat / (k - 1)) / (sc_error / (N - k))
print(f"SCTrat = {sc_trat:.2f}  SCE = {sc_error:.2f}  F = {F:.2f}  p = {stats.f.sf(F, k - 1, N - k):.4f}")

# con statsmodels (mecánica como nivel de referencia)
modelo_ejemplo = smf.ols("minutos ~ C(bicicleta, Treatment(reference='mecánica'))", data=df).fit()
tabla_ejemplo = sm.stats.anova_lm(modelo_ejemplo, typ=1)
print(tabla_ejemplo)

## 2. Ejemplo · la forma matricial $\hat\beta = (X^\top X)^{-1} X^\top y$

In [ ]:
niveles = ["mecánica", "eléctrica", "de carga"]          # el primero es la referencia
X = np.column_stack([np.ones(N)] + [(df.bicicleta == g).to_numpy(float) for g in niveles[1:]])
y = df.minutos.to_numpy(float)
beta_ejemplo = np.linalg.solve(X.T @ X, X.T @ y)
print("X'X =\n", X.T @ X)
print("X'y =", X.T @ y)
print("beta =", beta_ejemplo)   # [media de mecánica, eléctrica − mecánica, de carga − mecánica]

## 3. Ejemplo · supuestos

In [ ]:
residuos = y - X @ beta_ejemplo
print("Shapiro-Wilk de los residuos:", stats.shapiro(residuos))
print("Levene (centrado en la mediana):", stats.levene(*grupos.values()))
print("Bartlett:", stats.bartlett(*grupos.values()))

## 4. Caso Estampa Sur · secado según tinta

In [ ]:
s = pd.read_csv(DATOS / "secado.csv")
print(s.groupby("tinta").secado_min.agg(["count", "mean", "var"]))
modelo_caso = smf.ols("secado_min ~ C(tinta)", data=s).fit()
tabla_caso = sm.stats.anova_lm(modelo_caso, typ=1)
print(tabla_caso)
print(modelo_caso.params)            # agua es la referencia (orden alfabético)
levene_caso = stats.levene(*[g.secado_min for _, g in s.groupby("tinta")])
bartlett_caso = stats.bartlett(*[g.secado_min for _, g in s.groupby("tinta")])
shapiro_caso = stats.shapiro(modelo_caso.resid)
print(levene_caso, bartlett_caso, shapiro_caso, sep="\n")
# los residuos según temperatura: lo que el modelo de un factor deja en el error
print(modelo_caso.resid.groupby(s.temperatura).mean())